# EFSM Phase 3 — QLoRA Fine-tuning

**Run on Kaggle with GPU T4 x2 selected.**

Before running:
- Add `HF_TOKEN` and `WANDB_API_KEY` as Kaggle Secrets (right panel → Add-ons → Secrets)
- Ensure your W&B project `efsm-cse465` exists at wandb.ai
- Ensure `tasbid001/efsm-checkpoints` model repo exists on HuggingFace

In [ ]:
# Cell 1 — Load secrets and authenticate
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

print('Secrets loaded and authenticated.')

In [ ]:
# Cell 2 — Clone repo and install requirements
import subprocess, os

REPO_URL = 'https://github.com/tasbidrahman10/empathetic-voice-llm.git'
REPO_DIR = 'empathetic-voice-llm'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    # Pull latest code if repo already exists
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Requirements installed.')

In [ ]:
# Cell 3 — Download preprocessed JSONL data from HuggingFace Hub
# The JSONL files were uploaded in Phase 2 under tasbid001/efsm-checkpoints
from huggingface_hub import hf_hub_download
import os

os.makedirs('data', exist_ok=True)
CHECKPOINT_REPO = 'tasbid001/efsm-checkpoints'

for split in ['train', 'val', 'test']:
    dest = f'data/{split}.jsonl'
    if not os.path.exists(dest):
        hf_hub_download(
            repo_id=CHECKPOINT_REPO,
            filename=f'data/{split}.jsonl',
            repo_type='model',
            local_dir='.',
            token=os.environ['HF_TOKEN'],
        )
        print(f'Downloaded {dest}')
    else:
        print(f'{dest} already present')

# Quick sanity check
for split in ['train', 'val', 'test']:
    path = f'data/{split}.jsonl'
    count = sum(1 for _ in open(path))
    print(f'{split}: {count} conversations')

In [ ]:
# Cell 4 — Run QLoRA fine-tuning
# Model is consolidated to GPU 0 — plain python, no accelerate launch needed.
# Monitor live at wandb.ai/efsm-cse465
import subprocess, os

env = os.environ.copy()
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(
    ['python', 'src/training/train.py', '--config', 'configs/config.yaml'],
    check=True,
    env=env,
)
print('Training complete. Return code:', result.returncode)

In [ ]:
# Cell 5 — Verify checkpoint files on HuggingFace Hub
from huggingface_hub import list_repo_files
import os

CHECKPOINT_REPO = 'tasbid001/efsm-checkpoints'
files = list(list_repo_files(CHECKPOINT_REPO, token=os.environ['HF_TOKEN']))
print('Files on HF Hub:')
for f in sorted(files):
    print(' ', f)